## Annualized ambitions

The data were cleaned to retain only unique entries for the baseline year and target year. A line was then used to connect baseline emissions to target-year emissions. The final annualized ambition was computed using the following function:
(1 + (baseline_value - target_year_emission) / baseline_value) ** (1 / (target_year - baseline_year)) - 1

In [3]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt


INPUT_PATH = Path("/Users/wheeinner/Library/Mobile Documents/com~apple~CloudDocs/UNC/problem_shifting/Ambition/data_raw/climactor_targets_combined.csv")
OUTPUT_PATH = Path("/Users/wheeinner/Library/Mobile Documents/com~apple~CloudDocs/UNC/problem_shifting/Ambition/data_inter/climactor_annualized_ambition.csv")
PLOT_PATH = Path("/Users/wheeinner/Library/Mobile Documents/com~apple~CloudDocs/UNC/problem_shifting/Ambition/data_inter/annualized_ambition_by_iso.png")
REQUIRED_COLUMNS = [
    "climactor_id",
    "baseline_value",
    "baseline_year",
    "target_year",
    "target_value",
]

## Select the ultimate target per climactor_id

In [4]:
df = pd.read_csv(INPUT_PATH)
original_rows = len(df)
df = df.drop_duplicates()

for column in REQUIRED_COLUMNS[1:]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df = df.dropna(subset=REQUIRED_COLUMNS).copy()
print(f"Read {original_rows} rows from {INPUT_PATH}.")
df.head()

latest_targets = (
    df.sort_values(["climactor_id", "target_year"], ascending=[True, False])
    .drop_duplicates(subset=["climactor_id"], keep="first")
    .copy()
)
latest_targets.head()

Read 9411 rows from /Users/wheeinner/Library/Mobile Documents/com~apple~CloudDocs/UNC/problem_shifting/Ambition/data_raw/climactor_targets_combined.csv.


,iso,entity_type,climactor_id,baseline_value,baseline_year,inv_year,total_emissions,target_year,target_value,target_record,GDAM_id
1,BEL,City,0033fb16f68dfeeb,2629824.000,2005,NaN,NaN,2050,100.0,0033fb16f68dfeeb_2050,BEL.2.1.1.2_1
2,JPN,SubRegion,004a0d7cb5606eef,1350000.000,2019,NaN,NaN,2030,10.0,004a0d7cb5606eef_2030,JPN.7.1159
3,BEL,City,00732a73f4f159b3,43491.300,2011,2011.0,43491.3,2030,40.0,00732a73f4f159b3_2030,BEL.2.2.1.5_1
5,ESP,City,00752000db04c515,11423.000,2005,2005.0,10036.9,2030,40.0,00752000db04c515_2030,ESP.09.08.08222
7,ITA,City,0083ef4ed71b869e,7739.785,2011,2011.0,7739.8,2030,40.0,0083ef4ed71b869e_2030,ITA.19.86.003


## Compute annualized ambition

In [5]:
# `target_value` is stored as a percentage
latest_targets["target_year_emission"] = latest_targets["baseline_value"] * (
    1 - latest_targets["target_value"] / 100.0
)

latest_targets["years_to_target"] = (
    latest_targets["target_year"] - latest_targets["baseline_year"]
)

annualized_base = 1 + (
    (latest_targets["baseline_value"] - latest_targets["target_year_emission"])
    / latest_targets["baseline_value"]
)

valid_mask = (
    (latest_targets["years_to_target"] > 0)
    & (latest_targets["baseline_value"] > 0)
    & (annualized_base >= 0)
)

latest_targets["annualized_ambition"] = np.nan
latest_targets.loc[valid_mask, "annualized_ambition"] = (
    np.power(
        annualized_base.loc[valid_mask],
        1 / latest_targets.loc[valid_mask, "years_to_target"],
    )
    - 1
)

output = latest_targets[
    [
        "iso",
        "climactor_id",
        "baseline_value",
        "baseline_year",
        "target_year",
        "target_value",
        "target_year_emission",
        "annualized_ambition",
    ]
].sort_values("climactor_id")

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
output.to_csv(OUTPUT_PATH, index=False)

print(f"Saved {len(output)} climactor rows to {OUTPUT_PATH}.")
print(
    "Annualized ambition is undefined for "
    f"{output['annualized_ambition'].isna().sum()} rows after validation."
)
output.head()

Saved 3657 climactor rows to /Users/wheeinner/Library/Mobile Documents/com~apple~CloudDocs/UNC/problem_shifting/Ambition/data_inter/climactor_annualized_ambition.csv.
Annualized ambition is undefined for 3 rows after validation.


,iso,climactor_id,baseline_value,baseline_year,target_year,target_value,target_year_emission,annualized_ambition
1,BEL,0033fb16f68dfeeb,2629824.000,2005,2050,100.0,0.000,0.015523
2,JPN,004a0d7cb5606eef,1350000.000,2019,2030,10.0,1215000.000,0.008702
3,BEL,00732a73f4f159b3,43491.300,2011,2030,40.0,26094.780,0.017867
5,ESP,00752000db04c515,11423.000,2005,2030,40.0,6853.800,0.013550
7,ITA,0083ef4ed71b869e,7739.785,2011,2030,40.0,4643.871,0.017867


## Plot the distribution of annualized ambition per iso

In [6]:
def save_plot(output: pd.DataFrame) -> None:
    plot_df = output.dropna(subset=["annualized_ambition"]).copy()
    if plot_df.empty:
        return

    order = (
        plot_df.groupby("iso")["annualized_ambition"]
        .median()
        .sort_values()
        .index
    )
    iso_counts = plot_df["iso"].value_counts()
    labeled_iso = [f"{iso} (n={iso_counts[iso]})" for iso in order]
    plot_df["iso_label"] = pd.Categorical(
        plot_df["iso"].map(dict(zip(order, labeled_iso))),
        categories=labeled_iso,
        ordered=True,
    )

    fig_height = max(10, len(order) * 0.35)
    fig, ax = plt.subplots(figsize=(12, fig_height))
    plot_df.boxplot(
        column="annualized_ambition",
        by="iso_label",
        ax=ax,
        vert=False,
        grid=False,
        showfliers=False,
    )
    ax.set_title("Distribution of Annualized Ambition by ISO")
    ax.set_xlabel("Annualized Ambition")
    ax.set_ylabel("ISO")
    ax.grid(axis="x", linestyle="--", alpha=0.4)
    fig.suptitle("")
    fig.tight_layout()
    fig.savefig(PLOT_PATH, dpi=200, bbox_inches="tight")
    plt.close(fig)


save_plot(output)
print(f"Saved ISO distribution plot to {PLOT_PATH}.")

Saved ISO distribution plot to /Users/wheeinner/Library/Mobile Documents/com~apple~CloudDocs/UNC/problem_shifting/Ambition/data_inter/annualized_ambition_by_iso.png.
